# Notebook 02 — Feature Engineering
## AeroTwinML · Multi-City Features

**Objective:** Transform raw merged data into ML-ready features using `FeatureBuilder`.

**Feature groups:**
1. **Time features** — hour, day, month, cyclical encodings
2. **City features** — encoded city identifier (0=Hyderabad, 1=Karachi)
3. **Lag features** — AQI at t-1, t-6, t-24, t-72
4. **Rolling features** — 6h and 24h rolling mean/std
5. **Weather features** — raw weather variables
6. **Interaction features** — humidity*temp, wind*pm25, etc.
7. **Targets** — AQI at t+24h, t+48h, t+72h (shifted)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils.config import get
from utils.storage import load_parquet
from feature_store.feature_builder import FeatureBuilder

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

# Load merged data
DATA_DIR = Path(get('storage.data_dir', '../data'))
merged_path = DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet'

df = load_parquet(merged_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Input: {len(df)} rows, {len(df.columns)} columns')
print(f'Cities: {df["city"].unique().tolist() if "city" in df.columns else "single"}')
df.head()

## 1. Build All Features

In [ ]:
builder = FeatureBuilder(df)
featured = builder.build_all()

print(f'Output: {len(featured)} rows, {len(featured.columns)} columns')
print(f'\nNew columns added:')
new_cols = [c for c in featured.columns if c not in df.columns]
for col in new_cols:
    print(f'  {col}')

## 2. Time Features

In [ ]:
time_cols = ['hour', 'day', 'day_of_week', 'month', 'weekend', 'season',
             'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
available = [c for c in time_cols if c in featured.columns]

if available:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Hour distribution
    featured['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 0], color='#00b4d8')
    axes[0, 0].set_title('Distribution of Hours')
    
    # Cyclical hour encoding
    axes[0, 1].scatter(featured['hour_sin'], featured['hour_cos'], c=featured['hour'], cmap='viridis', s=1)
    axes[0, 1].set_title('Cyclical Hour Encoding')
    axes[0, 1].set_xlabel('hour_sin')
    axes[0, 1].set_ylabel('hour_cos')
    
    # Weekend vs weekday
    if 'aqi' in featured.columns:
        featured.groupby('weekend')['aqi'].mean().plot(kind='bar', ax=axes[1, 0], color=['#4da6ff', '#ff7e00'])
        axes[1, 0].set_xticklabels(['Weekday', 'Weekend'], rotation=0)
        axes[1, 0].set_title('Average AQI: Weekday vs Weekend')
    
    # Season
    if 'aqi' in featured.columns:
        featured.groupby('season')['aqi'].mean().plot(kind='bar', ax=axes[1, 1], color='#00e400')
        axes[1, 1].set_xticklabels(['Winter', 'Spring', 'Summer', 'Autumn'], rotation=45)
        axes[1, 1].set_title('Average AQI by Season')
    
    plt.tight_layout()
    plt.show()

## 3. City Feature (Multi-City Encoding)

In [ ]:
if 'city_encoded' in featured.columns and 'city' in featured.columns:
    city_map = featured.groupby('city')['city_encoded'].first()
    print('City encoding:')
    for city, code in city_map.items():
        count = (featured['city'] == city).sum()
        print(f'  {city} -> {code} ({count} rows)')
    
    # Verify encoding is consistent
    print(f'\ncity_encoded values: {sorted(featured["city_encoded"].unique())}')
else:
    print('No city_encoded feature (single city data)')

## 4. Lag Features

In [ ]:
lag_cols = [c for c in featured.columns if 'lag' in c.lower() or 'aqi_lag' in c]

if lag_cols:
    print(f'Lag features ({len(lag_cols)}):')
    for col in lag_cols[:10]:
        non_null = featured[col].notna().sum()
        print(f'  {col}: {non_null}/{len(featured)} non-null, mean={featured[col].mean():.1f}')
    
    # Correlation of lags with target
    if 'target_aqi_24h' in featured.columns:
        lag_corr = featured[lag_cols + ['target_aqi_24h']].corr()['target_aqi_24h'].drop('target_aqi_24h')
        
        fig, ax = plt.subplots(figsize=(12, 5))
        lag_corr.sort_values().plot(kind='barh', ax=ax, color='#00b4d8')
        ax.set_title('Lag Feature Correlation with 24h Target')
        ax.set_xlabel('Correlation')
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.show()

## 5. Target Construction

Targets are future AQI values shifted by 24h, 48h, 72h.

In [ ]:
target_cols = [c for c in featured.columns if c.startswith('target_aqi_')]

if target_cols:
    print('Target columns:')
    for col in target_cols:
        non_null = featured[col].notna().sum()
        mean_val = featured[col].mean()
        print(f'  {col}: {non_null}/{len(featured)} non-null, mean={mean_val:.1f}')
    
    # Training data (rows with all targets)
    train_df = builder.get_training_data()
    print(f'\nTraining rows (with targets): {len(train_df)}')
    
    # Target distribution
    fig, axes = plt.subplots(1, len(target_cols), figsize=(5*len(target_cols), 5))
    if len(target_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, target_cols):
        train_df[col].hist(bins=50, ax=ax, color='#00b4d8', alpha=0.7)
        ax.set_title(f'{col} Distribution')
        ax.set_xlabel('AQI')
    plt.tight_layout()
    plt.show()

## 6. Feature Importance Preview (Correlation with Target)

In [ ]:
if 'target_aqi_24h' in featured.columns:
    # Get numeric feature columns
    feature_cols = [c for c in featured.columns if not c.startswith('target_')
                    and c not in ('timestamp', 'source', 'station_name', 'city', 'country',
                                  'dominant_pollutant', 'merged_at', 'fetched_at')
                    and featured[c].dtype in ('float64', 'float32', 'int64', 'int32')]
    
    corr = featured[feature_cols + ['target_aqi_24h']].corr()['target_aqi_24h'].drop('target_aqi_24h')
    top_corr = corr.abs().sort_values(ascending=False).head(15)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['#00e400' if corr[f] > 0 else '#ff3333' for f in top_corr.index]
    top_corr.plot(kind='barh', ax=ax, color=colors)
    ax.set_title('Top 15 Features Correlated with 24h AQI Target')
    ax.set_xlabel('|Correlation|')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
    
    print(f'Total feature columns: {len(feature_cols)}')

## Summary

| Feature Group | Count | Description |
|---------------|-------|-------------|
| Time | ~12 | hour, day, month, cyclical encodings, weekend, season |
| City | 1 | city_encoded (0/1 for multi-city) |
| Lag | ~8 | AQI at t-1, t-6, t-24, t-72 hours |
| Rolling | ~4 | 6h and 24h rolling mean/std of AQI |
| Weather | ~8 | temperature, humidity, wind, pressure, precipitation, cloud_cover |
| Interaction | ~4 | humidity*temp, wind*pm25, rain*pm10, aqi_change_rate |
| Target | 3 | AQI at t+24h, t+48h, t+72h |

**Next:** Notebook 03 — Per-Horizon Model Training